**Hard vs. Soft Voting**
- **Dataset:** from sklearn.datasets import fetch_california_housing
- **Objective:** Learn to combine heterogenous models.
- **Task:** Implement a VotingClassifier using scikit-learn.
- **Requirements:**  
              - Select three fundamentally different algorithms (e.g., Logistic Regression, Support Vector Machine, and Random Forest).  
              - Evaluate the ensemble using Hard Voting (majority rule).  
              - Evaluate the ensemble using Soft Voting (averaging predicted probabilities). Note: Ensure the SVM is configured to output probabilities.  
              - Compare the ensemble's performance against the best individual base model.

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# 1. Load data and convert to a binary classification task
data = fetch_california_housing()
X, y_continuous = data.data, data.target

# Binarize target: 1 if above median value, 0 otherwise
median_val = np.median(y_continuous)
y = (y_continuous > median_val).astype(int)

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features (crucial for Logistic Regression and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Define three fundamentally different base algorithms
# Note: probability=True is required for SVM to perform soft voting
clf1 = LogisticRegression(random_state=42, max_iter=1000)
clf2 = SVC(probability=True, random_state=42)
clf3 = RandomForestClassifier(random_state=42)

# 3. Evaluate individual base models to find the best performer
base_models = {'Logistic Regression': clf1, 'SVM': clf2, 'Random Forest': clf3}
best_base_name = ""
best_base_score = 0

print("--- Individual Base Models Performance ---")
for name, clf in base_models.items():
    clf.fit(X_train_scaled, y_train)
    score = clf.score(X_test_scaled, y_test)
    print(f"{name} Accuracy: {score:.4f}")
    if score > best_base_score:
        best_base_score = score
        best_base_name = name

# 4. Implement and evaluate Hard Voting Ensemble
ensemble_hard = VotingClassifier(
    estimators=[('lr', clf1), ('svm', clf2), ('rf', clf3)], 
    voting='hard'
)
ensemble_hard.fit(X_train_scaled, y_train)
hard_score = ensemble_hard.score(X_test_scaled, y_test)

# 5. Implement and evaluate Soft Voting Ensemble
ensemble_soft = VotingClassifier(
    estimators=[('lr', clf1), ('svm', clf2), ('rf', clf3)], 
    voting='soft'
)
ensemble_soft.fit(X_train_scaled, y_train)
soft_score = ensemble_soft.score(X_test_scaled, y_test)

# 6. Final Comparison Results
print("\n--- Ensemble Comparison Summary ---")
print(f"Best Individual Model ({best_base_name}): {best_base_score:.4f}")
print(f"Hard Voting Ensemble Accuracy:       {hard_score:.4f}")
print(f"Soft Voting Ensemble Accuracy:       {soft_score:.4f}")

--- Individual Base Models Performance ---
Logistic Regression Accuracy: 0.8272
SVM Accuracy: 0.8603
Random Forest Accuracy: 0.8941

--- Ensemble Comparison Summary ---
Best Individual Model (Random Forest): 0.8941
Hard Voting Ensemble Accuracy:       0.8651
Soft Voting Ensemble Accuracy:       0.8729


**The Sklearn Stacking Framework**
- **Dataset:** from sklearn.datasets import fetch_california_housing
- **Objective:** Understand how meta-learners operate on the predictions of base learners.
- **Task:** Build a two-level StackingClassifier or StackingRegressor.
- **Requirements:**  
              - Level-0 (Base Models): Use a mix of models (e.g., Ridge Regression, Random Forest, K-Nearest Neighbors).  
              - Level-1 (Meta-Model): Use a simple, interpretable model (e.g., Linear/Logistic Regression) to learn the weights of the level-0 models.  
              - Inspect the coefficients of the Level-1 Linear Regression model. Which Level-0 model was trusted the most by the meta-learner?


In [2]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import StackingRegressor

# Load dataset
data = fetch_california_housing()
X, y = data.data, data.target

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features (essential for Ridge and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define Level-0 Base Models
base_models = [
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)),
    ('knn', KNeighborsRegressor(n_neighbors=5))
]

# Define Level-1 Meta-Model
meta_model = LinearRegression()

# Build the Stacking Framework
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5, # 5-fold cross-validation for generating meta-features
    n_jobs=-1
)

# Train the framework
stacking_model.fit(X_train_scaled, y_train)

# Evaluate performance (R² Score)
print("--- Model Performance ($R^2$ Score) ---")
for name, model in stacking_model.named_estimators_.items():
    score = model.score(X_test_scaled, y_test)
    print(f"Base Model - {name.upper()}: {score:.4f}")

stacking_score = stacking_model.score(X_test_scaled, y_test)
print(f"Stacking Regressor: {stacking_score:.4f}\n")

# Inspect Level-1 Coefficients
meta_coefs = stacking_model.final_estimator_.coef_
print("--- Level-1 Meta-Learner Coefficients ---")
for (name, _), coef in zip(base_models, meta_coefs):
    print(f"Weight assigned to {name.upper()}: {coef:.4f}")

--- Model Performance ($R^2$ Score) ---
Base Model - RIDGE: 0.5958
Base Model - RF: 0.8034
Base Model - KNN: 0.6728
Stacking Regressor: 0.8046

--- Level-1 Meta-Learner Coefficients ---
Weight assigned to RIDGE: -0.0038
Weight assigned to RF: 0.9296
Weight assigned to KNN: 0.1067
